<a href="https://colab.research.google.com/github/gemsinlife30/gemcode-demo/blob/main/practical6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.width", 100)

# ---------------------------------------------------------------
# Load data directly from the web (no upload needed in Colab)
# ---------------------------------------------------------------
url = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/TeachingRatings.csv"
df = pd.read_csv(url)
df = df.drop(columns=["rownames"])
print("Shape:", df.shape)
print(df.head(), "\n")

# =================================================================
# Q1. T-TEST: Does gender affect teaching evaluation score?
# =================================================================
print("=" * 70)
print("Q1. INDEPENDENT SAMPLES T-TEST -- gender vs eval score")
print("=" * 70)

male_eval = df.loc[df["gender"] == "male", "eval"]
female_eval = df.loc[df["gender"] == "female", "eval"]

print(f"Male   : n={len(male_eval)}, mean={male_eval.mean():.4f}, sd={male_eval.std():.4f}")
print(f"Female : n={len(female_eval)}, mean={female_eval.mean():.4f}, sd={female_eval.std():.4f}")

levene_stat, levene_p = stats.levene(male_eval, female_eval)
print(f"\nLevene's test for equal variances: stat={levene_stat:.4f}, p={levene_p:.4f}")
equal_var = levene_p > 0.05
print(f"Assume equal variances: {equal_var}")

t_stat, t_p = stats.ttest_ind(male_eval, female_eval, equal_var=equal_var)
print(f"\nIndependent t-test: t={t_stat:.4f}, p={t_p:.4f}")

pooled_sd = np.sqrt(((len(male_eval)-1)*male_eval.std()**2 + (len(female_eval)-1)*female_eval.std()**2) /
                     (len(male_eval)+len(female_eval)-2))
cohens_d = (male_eval.mean() - female_eval.mean()) / pooled_sd
print(f"Cohen's d = {cohens_d:.4f}")

alpha = 0.05
print(f"\nDecision: {'REJECT H0 -> significant gender effect' if t_p < alpha else 'FAIL TO REJECT H0 -> no significant gender effect'}")

# =================================================================
# Q2. ANOVA: Does beauty score differ by age group?
# =================================================================
print("\n" + "=" * 70)
print("Q2. ONE-WAY ANOVA -- age group vs beauty score")
print("=" * 70)

bins = [0, 40, 50, 60, 100]
labels = ["<=40", "41-50", "51-60", "60+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels)

print(df.groupby("age_group", observed=True)["beauty"].agg(["count", "mean", "std"]), "\n")

groups = [g["beauty"].values for _, g in df.groupby("age_group", observed=True)]
f_stat, anova_p = stats.f_oneway(*groups)
print(f"One-way ANOVA: F={f_stat:.4f}, p={anova_p:.4f}")
print(f"Decision: {'REJECT H0 -> beauty score differs across age groups' if anova_p < alpha else 'FAIL TO REJECT H0'}")

age_beauty_r, age_beauty_p = stats.pearsonr(df["age"], df["beauty"])
print(f"\n(Context) Pearson correlation age vs beauty: r={age_beauty_r:.4f}, p={age_beauty_p:.4f}")

# =================================================================
# Q3. CHI-SQUARE: Is there an association between tenure and gender?
# =================================================================
print("\n" + "=" * 70)
print("Q3. CHI-SQUARE TEST OF INDEPENDENCE -- tenure vs gender")
print("=" * 70)

contingency = pd.crosstab(df["gender"], df["tenure"])
print("Contingency table (observed counts):")
print(contingency, "\n")

chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi-square = {chi2:.4f}, df = {dof}, p = {chi_p:.4f}")
print("Expected counts:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(2))

n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))
print(f"\nCramer's V = {cramers_v:.4f}")
print(f"\nDecision: {'REJECT H0 -> tenure and gender are associated' if chi_p < alpha else 'FAIL TO REJECT H0 -> no significant association'}")

# =================================================================
# Q4. CORRELATION: Is eval score correlated with beauty score?
# =================================================================
print("\n" + "=" * 70)
print("Q4. PEARSON CORRELATION -- eval score vs beauty score")
print("=" * 70)

r, corr_p = stats.pearsonr(df["eval"], df["beauty"])
print(f"Pearson r = {r:.4f}, p = {corr_p:.6f}, n = {len(df)}")
r_squared = r ** 2
print(f"R-squared = {r_squared:.4f}  (beauty explains ~{r_squared*100:.1f}% of variance in eval score)")

print(f"\nDecision: {'REJECT H0 -> significant correlation' if corr_p < alpha else 'FAIL TO REJECT H0'}")
if corr_p < alpha:
    direction = "positive" if r > 0 else "negative"
    strength = "weak" if abs(r) < 0.3 else "moderate" if abs(r) < 0.5 else "strong"
    print(f"Relationship is {direction} and {strength} in strength.")

print("\nDone.")

Shape: (463, 12)
  minority  age  gender credits    beauty  eval division native tenure  students  allstudents  \
0      yes   36  female    more  0.289916   4.3    upper    yes    yes        24           43   
1       no   59    male    more -0.737732   4.5    upper    yes    yes        17           20   
2       no   51    male    more -0.571984   3.7    upper    yes    yes        55           55   
3       no   40  female    more -0.677963   4.3    upper    yes    yes        40           46   
4       no   31  female    more  1.509794   4.4    upper    yes    yes        42           48   

   prof  
0     1  
1     2  
2     3  
3     4  
4     5   

Q1. INDEPENDENT SAMPLES T-TEST -- gender vs eval score
Male   : n=268, mean=4.0690, sd=0.5567
Female : n=195, mean=3.9010, sd=0.5388

Levene's test for equal variances: stat=0.1194, p=0.7298
Assume equal variances: True

Independent t-test: t=3.2499, p=0.0012
Cohen's d = 0.3059

Decision: REJECT H0 -> significant gender effect

Q2. ONE-